# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
3908,30010,JERRICK'S LOWER EAST SIDE,1545 RTE 52,FISHKILL,12524,0,DRUG STORE,-73.87416,41.54451,"b'\x01\x01\x00\x00\x00+\xde\xc8<\xf2wR\xc0(,\xf1\x80\xb2\xc5D@'",360270603012,36027060301
14974,114777,EL TRONCO FOOD CORP,853 CLARKSON AVE,BROOKLYN,11203,0,GROCERY STORE,-73.92974,40.65669,b'\x01\x01\x00\x00\x00\x1dr3\xdc\x80{R\xc0\x1c\xce\xfcj\x0eTD@',360470868001,36047086800
1391,74117,RITE AID #1912,3987 LOCKPORT OLCOTT RD,LOCKPORT,14094,296,DRUG STORE,-78.67621,43.21966,b'\x01\x01\x00\x00\x00\x9a\xceN\x06G\xabS\xc0\xa3\x1e\xa2\xd1\x1d\x9cE@',360630242012,36063024201
12456,110138,CITY SMOKE INC,476 6TH AVE,NEW YORK,10011,0,CONV. STORE,-73.99797,40.73580,b'\x01\x01\x00\x00\x00\xe7\x18\x90\xbd\xde\x7fR\xc0\xca2\xc4\xb1.^D@',360610063002,36061006300
771,113719,STAR FOOD MART AND DELI CORP,301 E 4TH ST,NEW YORK,10009,0,CONV. STORE,-73.97997,40.72220,b'\x01\x01\x00\x00\x00\xe9C\x17\xd4\xb7~R\xc0\xea\x95\xb2\x0cq\\D@',360610026011,36061002601


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
9566,48357,769620.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2108.547945,2108.547945,False,2108.547945
13353,90198,1460449.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,335.0,365,365.0,0.0,0.0,4001.231507,4001.231507,False,4001.231507
4068,111585,467920.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,363.0,365,365.0,0.0,0.0,1281.972603,1281.972603,False,1281.972603
10404,64406,440220.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,1206.082192,1206.082192,False,1206.082192
5287,113608,5086134.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,13934.615068,13934.615068,False,13934.615068


## Overture Maps Data

In [7]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df['highest_hierarchy'] = interim_overture_df['taxonomy_hierarchy'].str[0]
print(interim_overture_df['highest_hierarchy'].unique())
print(interim_overture_df['basic_category'].nunique())
interim_overture_df.sample(20)

['sports_and_recreation' 'lifestyle_services' 'community_and_government'
 'services_and_business' 'food_and_drink' 'shopping'
 'travel_and_transportation' 'education' 'health_care' 'lodging'
 'arts_and_entertainment' 'cultural_and_historic' 'geographic_entities'
 None]
207


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen,highest_hierarchy
3562,d5e2ba0e-02d9-4e24-8eb9-d7f26986c39c,b'\x01\x01\x00\x00\x00k1\xff\x00\xd4rR\xc0\x8e\xd4\x9a\xf1\x87 E@',-73.794190,42.254149,bed_and_breakfast,bed_and_breakfast,"[lodging, bed_and_breakfast]",0.986759,NaN,2025-06-25.0,lodging
512,4f1d5b06-08cc-4888-a1da-1f06e826836b,b'\x01\x01\x00\x00\x00\t^\xa1\xcd9)R\xc0r\xc3\xad\x8a\x89gD@',-72.644153,40.808885,personal_or_beauty_service,barber,"[lifestyle_services, personal_or_beauty_service, barber]",0.990219,NaN,2025-06-25.0,lifestyle_services
4493,58c32927-8806-4534-8420-993c1a71c162,"b""\x01\x01\x00\x00\x00\xbf'{\xf0\x7fsR\xc07\x0fP\xafN\x87E@""",-73.804684,43.057089,museum,museum,"[arts_and_entertainment, museum]",0.990219,NaN,2025-06-25.0,arts_and_entertainment
426,36174cae-283a-40b3-8249-650323b39612,b'\x01\x01\x00\x00\x00\x88\xb0m\x93\xc2*R\xc06\xd7a3\xdduD@',-72.668126,40.920813,brewery,brewery,"[food_and_drink, alcoholic_beverage_venue, brewery]",0.990219,NaN,2025-06-25.0,food_and_drink
1506,c213e6d1-0691-44da-8e91-ce36936e4c4c,b'\x01\x01\x00\x00\x00\xa3\x95{\x81YvR\xc0\xaf\xb5\xf7\xa9*XE@',-73.849213,42.688802,electronics_store,electronics_store,"[shopping, specialty_store, electronics_store]",0.990219,Apple,2025-06-25.0,shopping
1460,30853037-0df6-4277-a9db-c19961c515f5,b'\x01\x01\x00\x00\x00\xdf\xfb\x1b\xb4WvR\xc0a\xfd\x9f\xc3|]E@',-73.849103,42.730370,office_supply_store,office_supply_store,"[shopping, specialty_store, office_supply_store]",0.965500,NaN,2025-06-25.0,shopping
2935,89f8e183-53e2-4398-9579-df708ee6e555,"b""\x01\x01\x00\x00\x00R\xff\x0ff\xc9pR\xc0!\xd9ma'^E@""",-73.762292,42.735577,electronics_store,mobile_phone_accessory_store,"[shopping, specialty_store, electronics_store, mobile_phone_accessory_store]",0.996895,NaN,2025-06-25.0,shopping
1149,b0662342-e1b6-4e76-94ff-468423cfa40e,b'\x01\x01\x00\x00\x00FJ\x1b\xe5+xR\xc0a`\xd9g\x8a\xf5D@',-73.877679,41.918286,park,park,"[sports_and_recreation, park]",0.972216,NaN,2025-06-25.0,sports_and_recreation
612,025e92e5-911e-4624-b0d9-b7305e931b3c,b'\x01\x01\x00\x00\x00\xff\x85\xb3\x95\xf6.R\xc0\x1dg\x05L\x01jD@',-72.733800,40.828165,food_and_beverage_store,grocery_store,"[shopping, food_and_beverage_store, grocery_store]",0.990219,King Kullen,2025-06-25.0,shopping
2806,156a2eb8-a581-477b-b40e-0fc0a819b982,b'\x01\x01\x00\x00\x00u4\xf5W[qR\xc0\x85\x90V)\xfbTE@',-73.771200,42.663915,christian_place_of_worship,roman_catholic_place_of_worship,"[cultural_and_historic, place_of_worship, christian_place_of_worship, roman_catholic_place_of_worship]",0.957660,NaN,2025-06-25.0,cultural_and_historic


In [8]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [9]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
12,99013,FEEL GOOD PHARMACY INC,3058 3RD AVE,BRONX,10451,-73.91268,40.82070,b'\x01\x01\x00\x00\x00\xa6\xed_YizR\xc0\xe2\xe9\x95\xb2\x0ciD@',360050069001,36005006900,3,11,8,17,9,20,21,72,5,28,1,10
74,111311,ADK FOOD & FUEL,1 COUNTRY MILE,FRANKFORT,13340,-75.08829,43.03850,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6\xc5R\xc0\xb0rh\x91\xed\x84E@',360430102022,36043010202,0,2,1,1,0,0,0,0,0,0,0,1
30,83787,CASTLE GOURMET DELI CORP,2170 WESTCHESTER AVE,BRONX,10462,-73.85196,40.83418,b'\x01\x01\x00\x00\x00\x03`<\x83\x86vR\xc0\x18}\x05i\xc6jD@',360050222004,36005022200,2,17,1,1,6,12,6,25,3,7,0,5
62,81101,SOFT TOUCH NEWSTAND,1 METROTECH CENTER,BROOKLYN,11201,-73.98723,40.69376,b'\x01\x01\x00\x00\x00S\xe8\xbc\xc6.\x7fR\xc0\xf0\xa2\xaf \xcdXD@',360470011001,36047001100,1,3,4,17,14,26,49,96,19,42,10,21
58,112604,IN N OUT CONVENIENCE & SMOKE S,343 ROCKAWAY TPKE UNIT 1,LAWRENCE,11559,-73.73817,40.62905,b'\x01\x01\x00\x00\x00\xda8b->oR\xc0?\xc6\xdc\xb5\x84PD@',360594110001,36059411000,6,7,0,1,3,7,0,5,1,5,1,3


## LEHD Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
97,28158,JUNCTION FOOD BAZAAR,34-20 JUNCTION BLVD,JACKSON HEIGHTS,11372,-73.87274,40.75461,b'\x01\x01\x00\x00\x00Nz\xdf\xf8\xdawR\xc0\x06\x9e{\x0f\x97`D@',360810273022,36081027302,569.0,93.0,171.0,305.0
19,105943,129 GOURMET DELI & GROC CORP,129 E 45TH ST,NEW YORK,10017,-73.97442,40.75327,b'\x01\x01\x00\x00\x00_$\xb4\xe5\\~R\xc0j\x87\xbf&k`D@',360610092001,36061009200,909.0,117.0,67.0,725.0


In [12]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
82,103275,AAA EAST MEADOW GAS INC,1878 HEMPSTEAD TPKE,EAST MEADOW,11554,-73.56688,40.72382,"b'\x01\x01\x00\x00\x00j0\r\xc3GdR\xc0j\x18>""\xa6\\D@'",360594080001,36059408000,291.0,28.0,114.0,91.0,105.0,95.0
65,89815,AL'S DELI & GROCERY I CORP,160-26 BAISLEY BLVD,JAMAICA,11434,-73.78143,40.67699,b'\x01\x01\x00\x00\x00@\x87\xf9\xf2\x02rR\xc0\n\xdc\xba\x9b\xa7VD@',360810294001,36081029400,96.0,69.0,3.0,24.0,34.0,38.0


## AADT Data

In [8]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [12]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

9759 16801


,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
6434,48515,CLIFF'S LOCAL MARKET #514,11468 STATE RT 12,ALDER CREEK,13301,-75.224397,43.415909,360650242001,36065024200,54296.0,295.0,EGYPT RD,09,1:1||||,872.374730
7488,114280,FULL CIRCLE BAR AND GRILL,616 LERAY ST,WATERTOWN,13601,-75.912530,43.983210,360450614004,36045061400,NaN,NaN,NaN,NaN,NaN,NaN
16242,111938,LIGHTHOUSE PHARMACY,747 MONTAUK HWY,PATCHOGUE,11772,-72.987360,40.767720,361031591103,36103159110,NaN,NaN,NaN,NaN,NaN,NaN
10137,64336,SPEEDWAY #07804,550 ATLANTIC AVE,OCEANSIDE,11572,-73.656080,40.641120,360594130013,36059413001,NaN,NaN,NaN,NaN,NaN,NaN
4613,113734,7-ELEVEN #18249F,266 HOOKER AVENUE,POUGHKEEPSIE,12603,-73.909950,41.686260,360272209014,36027220901,55064.0,1940.0,S GRAND AVE,19,1:1||||,51.836833
1215,110976,RS FOOD MART INC,804 LAKE ST,ELMIRA,14901,-76.804140,42.100260,360150001003,36015000100,54832.0,9214.0,E Water St,14,1:1||||,1346.713675
4690,80462,STOP & SHOP #0503,5716 BROADWAY,BRONX,10463,-73.902010,40.881870,360050283002,36005028300,54438.0,1816.0,W 238TH ST,17,1:1||||,354.356161
11348,77789,MIRABITO #83-GLEN AUBREY,2965 NYS RT 26,GLEN AUBREY,13777,-76.035590,42.214440,360070102001,36007010200,NaN,NaN,NaN,NaN,NaN,NaN
2042,114942,C-TOWN,2807 86TH ST,BROOKLYN,11223,-73.978740,40.592810,360470308001,36047030800,55646.0,5927.0,BENSON AVE,16,1:1||||,506.110786
11475,79515,TOURE ENTERPRISE,905 COLUMBUS AVE,NEW YORK,10025,-73.963700,40.798120,360610189005,36061018900,NaN,NaN,NaN,NaN,NaN,NaN


## EPA Data

In [15]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [16]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
56,110788,SUNRISE 99 CENT STORE INC,4008 QUEENS BLVD,SUNNYSIDE,11104,0,VARIETY,-73.92392,40.74361,b'\x01\x01\x00\x00\x00\xb3\x07Z\x81!{R\xc0A}\x...,360810181013,36081018101,162297,36,81,18101,3,73.247865,137.816070,33.448447,2.857937,2.857937,24.345389,1.376044,1.270194,0.529248,0.211699,0.0,39.499071,12.500000,36,081,018101,3
20,113105,NAZ DELI CORP,234 E 106TH ST,NEW YORK,10029,0,SUPERMARKETS,-73.94326,40.79098,"b""\x01\x01\x00\x00\x00\xff\xe70_^|R\xc0!\xe5'\...",360610170001,36061017000,153070,36,61,17000,1,56.809486,105.767254,30.560116,2.309329,6.235187,0.153955,1.616530,0.000000,18.628585,1.616530,0.0,17.201267,10.666667,36,061,017000,1
64,113075,RAINBOW DELI GROCERY CORP,38-15 GREENPOINT AVE,LONG ISLAND CITY,11101,0,DELI-GROCERY,-73.92779,40.73838,b'\x01\x01\x00\x00\x00\x93\xe3N\xe9`{R\xc0\x9f...,360810189003,36081018900,149243,36,81,18900,3,27.428359,70.809118,18.068534,0.081390,0.162780,1.220847,1.383626,0.244169,13.673485,1.302237,0.0,42.892341,15.833333,36,081,018900,3
28,115205,ABE GRILL & MINI MART INC,1160 YONKERS AVE,YONKERS,10704,0,CONV. STORE,-73.85342,40.91390,b'\x01\x01\x00\x00\x00&p\xebn\x9evR\xc0>\xe8\x...,361190015051,36119001505,153582,36,119,1505,1,19.960106,34.390499,2.125535,0.066423,0.232480,0.016606,0.381932,0.963133,0.000000,0.464961,0.0,30.773597,12.333333,36,119,001505,1
76,78437,CITY MARKET & DELI INC,1245 EAST DELAVAN AVE,BUFFALO,14215,0,CONV. STORE,-78.81093,42.92269,b'\x01\x01\x00\x00\x00\x19V\xf1F\xe6\xb3S\xc0R...,360290041003,36029004100,159797,36,29,4100,3,6.600410,15.409228,1.414374,0.769221,0.397017,0.000000,0.223322,0.024814,0.000000,0.000000,0.0,25.981046,14.166667,36,029,004100,3


## ACS Data

In [17]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [18]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
46,39785,DANIELS DELI,2554 SOUTH AVENUE STORE 2,WAPPINGERS FALLS,12590,0,DELI-GROCERY,-73.91967,41.59167,b'\x01\x01\x00\x00\x000G\x8f\xdf\xdbzR\xc0\x1d...,360273000004,36027300000,76723.0,519.0,1174.0,0.219761,0.476150,0.304089,0.026658,0.188272,0.160494,0.532609,0.103499,0.527938,0.472062,0.653179,0.052023,0.294798,0.052023
48,77604,GEORGIA DELI,272 GEORGIA ST,BUFFALO,14201,0,GROCERY STORE,-78.87996,42.89121,b'\x01\x01\x00\x00\x00qr\xbfCQ\xb8S\xc0 \xefU+...,360290071022,36029007102,NaN,480.0,675.0,0.198519,0.648889,0.152593,0.293051,0.250804,0.081994,0.459259,0.000000,0.233333,0.766667,0.214583,0.168750,0.616667,0.489583
75,113075,RAINBOW DELI GROCERY CORP,38-15 GREENPOINT AVE,LONG ISLAND CITY,11101,0,DELI-GROCERY,-73.92779,40.73838,b'\x01\x01\x00\x00\x00\x93\xe3N\xe9`{R\xc0\x9f...,360810189003,36081018900,77803.0,338.0,549.0,0.183971,0.577413,0.238616,0.089530,0.433515,0.100182,0.808743,0.090090,0.292899,0.707101,0.452663,0.124260,0.423077,0.556213
63,100319,MIA TRAVEL MULTISERVICE,5 E CLARK PLACE,BRONX,10452,0,TRAVEL AGENT,-73.91877,40.83828,b'\x01\x01\x00\x00\x00\xf0\xa2\xaf \xcdzR\xc0\...,360050221013,36005022101,65609.0,625.0,1319.0,0.388931,0.595906,0.015163,0.319920,0.143275,0.000000,0.653651,0.154867,0.000000,1.000000,0.872000,0.060800,0.067200,0.721600
57,108836,SHUNFA DISCOUNT INC,5309 CHURCH AVE,BROOKLYN,11203,0,GROCERY STORE,-73.92711,40.65235,b'\x01\x01\x00\x00\x00\xdar.\xc5U{R\xc0\xd6\xc...,360470866003,36047086600,NaN,512.0,1045.0,0.113876,0.736842,0.149282,0.233102,0.287368,0.116842,0.688910,0.145975,0.156250,0.843750,0.462891,0.025391,0.511719,0.767578


## Master Data Sample

In [16]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(16801, 69)
Index(['agtno', 'busnm', 'busaddr', 'buscity', 'buszip', 'chain', 'bustype',
       'longitude', 'latitude', 'geometry', 'bg_geoid', 'tract_geoid',
       'aadt_count', 'aadt_road_name', 'aadt_functional_class', 'aadt_lanes',
       'dist_to_read_m', 'median_household_income', 'households_total',
       'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'age_65_plus_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'renter_occupied_proportion', 'family_households_proportion',
       'nonfamily_multiperson_households_proportion',
       'living_alone_households_proportion',
       'no_vehicle_households_proportion', 'D1A', 'D1B', 'D1C', 'D1C8_RET',
       'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT', 'D1C8_ED', 'D1C8_HLTH',
       'D1C8_PUB', 'D3A', 'NatWalkInd', 'gas_station_within_0.

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
16321,116553,PARISH FOOD MART INC,2886 E MAIN ST,PARISH,13131,0,CONV. STORE,-76.12004,43.40535,b'\x01\x01\x00\x00\x00\x8f\x8d@\xbc\xae\x07S\x...,360750205022,36075020502,NaN,NaN,NaN,NaN,NaN,72389.0,690.0,1434.0,0.155509,0.602510,0.241980,0.119714,0.095275,0.043377,0.590909,0.082854,0.804348,0.195652,0.633333,0.115942,0.250725,0.092754,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,788.0,135.0,186.0,467.0,510.0,34.0,45.0,82.0,138.0,290.0
15702,115742,7-ELEVEN #35495D,42-02 QUEEN BLVD,SUNNYSIDE,11104,30,CONV. STORE,-73.92204,40.74315,b'\x01\x01\x00\x00\x00\xa5f\x0f\xb4\x02{R\xc0\...,360810181022,36081018102,NaN,NaN,NaN,NaN,NaN,89702.0,496.0,829.0,0.143546,0.782871,0.073583,0.042553,0.371728,0.244764,0.656036,0.105903,0.010081,0.989919,0.526210,0.120968,0.352823,0.745968,70.372511,114.276555,20.376518,8.822822,1.575504,1.890605,2.205706,3.256042,2.415773,0.210067,0.0,21.674202,14.666667,4.0,15.0,3.0,5.0,4.0,17.0,6.0,38.0,10.0,19.0,9.0,18.0,757.0,98.0,135.0,524.0,143.0,55.0,15.0,29.0,59.0,55.0
2580,89485,TSM GROCERY,2285 1ST AVE,NEW YORK,10035,0,GROCERY STORE,-73.93469,40.79701,b'\x01\x01\x00\x00\x00}y\x01\xf6\xd1{R\xc0\xe0...,360610188001,36061018800,57017.0,WILLIS AVE BRG,14,4:0||||,601.811277,27083.0,475.0,706.0,0.158640,0.480170,0.361190,0.300122,0.101983,0.123229,0.624646,0.088435,0.073684,0.926316,0.454737,0.000000,0.545263,0.981053,57.427891,101.081974,12.551937,0.111079,0.111079,0.000000,0.333237,1.444028,10.219276,0.333237,0.0,33.512819,8.333333,3.0,9.0,11.0,23.0,4.0,10.0,26.0,79.0,8.0,26.0,2.0,10.0,422.0,78.0,111.0,233.0,155.0,10.0,27.0,15.0,38.0,102.0
4401,36799,STEWART'S SHOPS #425,201 N COMRIE AVENUE,JOHNSTOWN,12095,23,CONV. STORE,-74.36083,43.01386,b'\x01\x01\x00\x00\x00\x9bZ\xb6\xd6\x17\x97R\x...,360359712001,36035971200,NaN,NaN,NaN,NaN,NaN,42200.0,554.0,964.0,0.258299,0.485477,0.256224,0.147929,0.100719,0.037170,0.476141,0.000000,0.925993,0.074007,0.339350,0.045126,0.615523,0.090253,0.588971,1.265471,1.959774,0.474737,0.244786,0.583036,0.142421,0.351602,0.004451,0.158740,0.0,12.444875,10.166667,3.0,4.0,0.0,0.0,2.0,2.0,1.0,3.0,0.0,3.0,0.0,3.0,542.0,94.0,159.0,289.0,1220.0,328.0,194.0,283.0,444.0,493.0
14659,114285,JULITIN DELI GROCERY CORP,1290 SUTTER AVE,BROOKLYN,11208,0,SUPERMARKETS,-73.87050,40.67298,b'\x01\x01\x00\x00\x00\xc1\xca\xa1E\xb6wR\xc0Y...,360471208011,36047120801,NaN,NaN,NaN,NaN,NaN,NaN,223.0,739.0,0.144790,0.687415,0.167794,0.000000,0.134494,0.058544,0.825440,0.000000,0.206278,0.793722,0.560538,0.152466,0.286996,0.860987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,7.0,1.0,5.0,4.0,7.0,4.0,12.0,1.0,11.0,0.0,3.0,340.0,64.0,111.0,165.0,10.0,6.0,0.0,5.0,5.0,0.0


## Missingness and Variance Check

In [31]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [32]:
numeric_features = ['aadt_count', 'dist_to_read_m',
                    'median_household_income', 'households_total', 'total_population',
                    'age_18_29_proportion', 'age_30_64_proportion', 'age_65_plus_proportion',
                    'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
                    'labor_force_participation_rate', 'unemployment_rate',
                    'owner_occupied_proportion', 'renter_occupied_proportion',
                    'family_households_proportion', 'nonfamily_multiperson_households_proportion',
                    'living_alone_households_proportion', 'no_vehicle_households_proportion',
                    'D1A', 'D1B', 'D1C', 'D1C8_RET', 'D1C8_OFF', 'D1C8_IND', 'D1C8_SVC', 'D1C8_ENT',
                    'D1C8_ED', 'D1C8_HLTH', 'D1C8_PUB', 'D3A', 'NatWalkInd',
                    'gas_station_within_0.5_mi', 'gas_station_within_1_mi', 'arts_and_entertainment_within_0.5_mi',
                    'arts_and_entertainment_within_1_mi', 'shopping_within_0.5_mi',
                    'shopping_within_1_mi', 'social_or_community_service_within_0.5_mi',
                    'social_or_community_service_within_1_mi', 'restaurant_within_0.5_mi',
                    'restaurant_within_1_mi', 'bar_within_0.5_mi', 'bar_within_1_mi',
                    'rac_c000', 'rac_ce01', 'rac_ce02', 'rac_ce03', 'wac_c000', 'wac_cns07',
                    'wac_cns18', 'wac_ce01', 'wac_ce02', 'wac_ce03']

categorical_features = ['chain', 'bustype', 'aadt_functional_class', 'aadt_lanes']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
aadt_count,58.085828,0.000000,3.180915,5466.000000,5865.659472,8804.543594,5334.000000,1.501032,563
dist_to_read_m,56.966847,0.000000,0.082988,627.332754,778.900103,445.966603,744.678223,0.572559,6606
D1A,18.314386,0.261889,0.320606,0.000000,17.706218,28.001488,22.008052,1.581449,6444
D1B,18.314386,0.202369,0.247741,0.000000,42.559408,64.675765,56.678018,1.519658,6450
D1C,18.314386,0.136897,0.167590,0.000000,30.103823,140.466582,11.276514,4.666071,6455
D1C8_RET,18.314386,4.821142,5.902069,0.000000,2.599771,10.403593,1.586033,4.001734,5926
D1C8_PUB,18.314386,66.728171,81.689012,0.000000,1.397457,20.067523,0.000000,14.360026,1000
D1C8_HLTH,18.314386,12.237367,14.981055,0.000000,4.933529,26.571133,1.387777,5.385827,5274
D1C8_ED,18.314386,40.955895,50.138444,0.000000,1.675417,12.355113,0.294076,7.374353,2937
D1C8_ENT,18.314386,9.713708,11.891577,0.000000,3.077399,14.310108,1.099186,4.650066,5432


In [54]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':              pct
 chain           
 0      74.822927
 30      3.345039
 23      2.208202
 46      1.720136
 296     1.410630
 ...          ...
 56      0.005952
 336     0.005952
 337     0.005952
 303     0.005952
 103     0.005952
 
 [106 rows x 1 columns]}

## Correlation

In [38]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951
